# Systems of Linear Equations

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 01.03 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/04_systems_of_equations.ipynb)

---

## 🎯 Learning Objective

Solve systems of linear equations with Gaussian elimination and LU decomposition, and understand what happens when a system is over- or under-determined.

---

## 📐 Theory

A system of linear equations is compactly written $A\mathbf{x} = \mathbf{b}$. Three cases
matter:

- **Square, well-determined** ($A$ is $n\times n$ and invertible): exactly one solution.
- **Overdetermined** (more equations than unknowns): generally *no* exact solution exists —
  instead we find the $\mathbf{x}$ that comes as close as possible.
- **Underdetermined** (fewer equations than unknowns): infinitely many solutions exist — we
  typically want the smallest one.

### Gaussian elimination

Gaussian elimination solves a square system by using row operations to reduce $A$ to upper
triangular form, then solving by back-substitution — the same manual technique from a first
algebra course, just done systematically. **LU decomposition** ($A = LU$) packages the
elimination steps themselves into a reusable factorization, so solving for many different
right-hand sides $\mathbf{b}$ becomes cheap after the one-time cost of factoring $A$.

### Least squares for overdetermined systems

When no exact solution exists, we minimize the squared residual:

$$\mathbf{x}^\star = \arg\min_{\mathbf{x}} \|A\mathbf{x}-\mathbf{b}\|_2^2$$

Setting the gradient of this objective to zero yields the **normal equations**:

$$A^\top A\, \mathbf{x}^\star = A^\top \mathbf{b} \qquad\Longrightarrow\qquad
\mathbf{x}^\star = (A^\top A)^{-1}A^\top \mathbf{b}$$

This is the exact closed-form solution behind ordinary least-squares linear regression.

---

## 👁️ Visual Intuition

A well-determined $2\times2$ system is the intersection point of two lines. An overdetermined
system (many data points, one line to fit) has no exact intersection — least squares finds the
line minimizing total squared vertical distance to every point.

In [ ]:
# Left: exact intersection of two lines. Right: least-squares best-fit line through noisy points.
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Square system: 2x + y = 5, x - y = 1  -> unique intersection
x_line = np.linspace(-1, 4, 100)
axes[0].plot(x_line, 5 - 2 * x_line, label="2x + y = 5")
axes[0].plot(x_line, x_line - 1, label="x - y = 1")
sol = np.linalg.solve(np.array([[2.0, 1.0], [1.0, -1.0]]), np.array([5.0, 1.0]))
axes[0].scatter(*sol, color="red", zorder=5, label=f"solution {np.round(sol, 2)}")
axes[0].set_title("Well-determined: unique intersection")
axes[0].legend(fontsize=8); axes[0].set_xlim(-1, 4); axes[0].set_ylim(-3, 6)

# Overdetermined system: many noisy points, no exact line fits all
rng = np.random.default_rng(0)
xs = np.linspace(0, 10, 25)
ys = 2.0 * xs + 1.0 + rng.normal(scale=1.5, size=xs.shape)
A = np.stack([xs, np.ones_like(xs)], axis=1)
w, resid, *_ = np.linalg.lstsq(A, ys, rcond=None)
axes[1].scatter(xs, ys, s=20, color="#4C72B0", label="data (no exact fit exists)")
axes[1].plot(xs, A @ w, color="#DD8452", label=f"least-squares fit: y={w[0]:.2f}x+{w[1]:.2f}")
axes[1].set_title("Overdetermined: least-squares best fit")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Gaussian elimination with back-substitution, implemented directly from the row-reduction
algorithm, then least squares via the normal equations.

In [ ]:
# Gaussian elimination + back-substitution, and least squares via the normal equations
import numpy as np

def gaussian_elimination_no_pivoting(A, b):
    """The naive textbook version: eliminate in column order, using whatever is on the
    diagonal as the pivot -- NO NUMERICAL SAFEGUARD. If that entry is zero, this divides
    by zero; if it is merely small, the elimination factor A[row,col]/pivot blows up and
    amplifies whatever rounding error is already in that row. Kept here ONLY as a
    contrast case for the pivoted version below -- do not use this on real problems."""
    A = A.copy().astype(float)
    b = b.copy().astype(float)
    n = len(b)
    for col in range(n):
        pivot = A[col, col]
        for row in range(col + 1, n):
            factor = A[row, col] / pivot
            A[row, :] -= factor * A[col, :]
            b[row] -= factor * b[col]
    x = np.zeros(n)
    for row in range(n - 1, -1, -1):
        x[row] = (b[row] - A[row, row + 1:] @ x[row + 1:]) / A[row, row]
    return x

def gaussian_elimination_solve(A, b):
    """Gaussian elimination WITH PARTIAL PIVOTING. Before eliminating column `col`, swap in
    whichever remaining row has the largest-magnitude entry in that column. This guarantees
    the pivot is never zero unless the matrix is genuinely singular, and keeps every
    elimination factor |A[row,col]/pivot| <= 1, which is exactly what prevents rounding
    error from being amplified step by step -- the standard fix `np.linalg.solve`'s own
    LAPACK backend already applies internally."""
    A = A.copy().astype(float)
    b = b.copy().astype(float)
    n = len(b)

    for col in range(n):
        pivot_row = col + np.argmax(np.abs(A[col:, col]))
        if pivot_row != col:
            A[[col, pivot_row], :] = A[[pivot_row, col], :]
            b[[col, pivot_row]] = b[[pivot_row, col]]

        pivot = A[col, col]
        for row in range(col + 1, n):
            factor = A[row, col] / pivot
            A[row, :] -= factor * A[col, :]
            b[row] -= factor * b[col]

    x = np.zeros(n)
    for row in range(n - 1, -1, -1):
        x[row] = (b[row] - A[row, row + 1:] @ x[row + 1:]) / A[row, row]
    return x

A = np.array([[2.0, 1.0, -1.0], [-3.0, -1.0, 2.0], [-2.0, 1.0, 2.0]])
b = np.array([8.0, -11.0, -3.0])

x_scratch = gaussian_elimination_solve(A, b)
x_numpy = np.linalg.solve(A, b)
print("Scratch solution:", np.round(x_scratch, 4))
print("np.linalg.solve :", np.round(x_numpy, 4))
print("Match:", np.allclose(x_scratch, x_numpy))

# --- Regression test: WHY pivoting matters. This system has a ZERO natural pivot in its
# top-left corner, which the no-pivoting version divides by directly. ---
A_zero_pivot = np.array([[0.0, 1.0], [1.0, 1.0]])
b_zero_pivot = np.array([1.0, 2.0])
true_x = np.array([1.0, 1.0])   # by inspection: 0*x0+1*x1=1 -> x1=1; x0+x1=2 -> x0=1

with np.errstate(divide='ignore', invalid='ignore'):
    x_no_pivot = gaussian_elimination_no_pivoting(A_zero_pivot, b_zero_pivot)
x_pivoted = gaussian_elimination_solve(A_zero_pivot, b_zero_pivot)

print("\n--- Regression test: a system with a ZERO natural pivot, A=[[0,1],[1,1]] ---")
print(f"No pivoting:   x = {x_no_pivot}  (divides by the exact zero at A[0,0] -> nan/inf)")
print(f"With pivoting: x = {np.round(x_pivoted, 8)}  (matches the true solution {true_x})")
print(f"Pivoted solution correct: {np.allclose(x_pivoted, true_x)}")
print(f"No-pivot solution is garbage (contains nan/inf): {np.any(~np.isfinite(x_no_pivot))}")

print("\n--- Least squares via the normal equations ---")
rng = np.random.default_rng(1)
n_samples, n_features = 40, 2
X = rng.standard_normal((n_samples, n_features))
true_w = np.array([1.5, -0.8])
y = X @ true_w + 0.1 * rng.standard_normal(n_samples)

# solve(), not inv(): forming the explicit inverse is both more expensive (an inverse is a
# full matrix, more work than one right-hand side needs) and less accurate -- solve() uses a
# pivoted LU factorization directly on X^T X rather than computing and then multiplying by
# a separately-rounded inverse matrix, one extra source of rounding error avoided.
w_normal_eq = np.linalg.solve(X.T @ X, X.T @ y)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
print("True weights          :", true_w)
print("Normal-equations fit   :", np.round(w_normal_eq, 4))
print("np.linalg.lstsq fit    :", np.round(w_lstsq, 4))

# --- Regression test: on an ILL-CONDITIONED design matrix (near-collinear columns), the
# explicit-inverse version loses noticeably more accuracy than solve(), and forming the
# normal equations at all (squaring the condition number, cond(X^T X) = cond(X)^2) loses
# more than going straight to lstsq's QR-based approach, which never forms X^T X. ---
n_ill = 30
x1_ill = rng.standard_normal(n_ill)
x2_ill = x1_ill + 1e-8 * rng.standard_normal(n_ill)     # nearly a duplicate of x1 -> near-collinear
X_ill = np.stack([x1_ill, x2_ill], axis=1)
true_w_ill = np.array([2.0, -1.0])
y_ill = X_ill @ true_w_ill + 1e-3 * rng.standard_normal(n_ill)

cond_X = np.linalg.cond(X_ill)
cond_XtX = np.linalg.cond(X_ill.T @ X_ill)
w_inv_ill = np.linalg.inv(X_ill.T @ X_ill) @ X_ill.T @ y_ill
w_solve_ill = np.linalg.solve(X_ill.T @ X_ill, X_ill.T @ y_ill)
w_lstsq_ill, *_ = np.linalg.lstsq(X_ill, y_ill, rcond=None)

# There is no single true answer here (the problem is genuinely near-singular: many (w1,w2)
# fit the data almost equally well), so we compare each estimate by its RESIDUAL -- how well
# it actually explains the data -- rather than by distance to true_w_ill, which is unstable
# to compare against when the columns are this collinear.
resid_inv = np.linalg.norm(X_ill @ w_inv_ill - y_ill)
resid_solve = np.linalg.norm(X_ill @ w_solve_ill - y_ill)
resid_lstsq = np.linalg.norm(X_ill @ w_lstsq_ill - y_ill)

print(f"\n--- Regression test: near-collinear design matrix (columns differ by ~1e-8) ---")
print(f"cond(X)     = {cond_X:.3e}")
print(f"cond(X^T X) = {cond_XtX:.3e}  (~cond(X)^2, exactly the squaring normal equations cause)")
print(f"Residual ||Xw-y|| via inv(X^T X)@X^T@y : {resid_inv:.6f}")
print(f"Residual ||Xw-y|| via solve(X^T X, ..) : {resid_solve:.6f}")
print(f"Residual ||Xw-y|| via lstsq (QR-based)  : {resid_lstsq:.6f}  (best-conditioned path: never forms X^T X)")

## 🤖 Why This Matters for AI

Ordinary least-squares linear regression **is** solving an overdetermined system with the
normal equations. This closed-form solution is the simplest possible instance of "training" —
minimizing a loss function over data — and every gradient-based optimizer in later modules is
solving a version of this same problem when a closed form doesn't exist.

Concretely, Ridge regression's closed-form $(X^\top X + \lambda I)^{-1}X^\top y$ is this exact computation with a regularization twist — `sklearn.linear_model.LinearRegression` calls LAPACK's least-squares solver internally. The reason closed-form solutions matter even in an age of gradient descent is that they are *exact* — no learning rate, no convergence to wait for — which is why linear models remain the standard baseline against which every gradient-trained model is compared.

---


In [ ]:
# Fitting a line to noisy data via the closed-form normal equations -- literally "training"
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n = 50
x = np.linspace(0, 10, n)
y_true_slope, y_true_intercept = 3.0, -2.0
y = y_true_slope * x + y_true_intercept + rng.normal(scale=2.0, size=n)

A = np.stack([x, np.ones(n)], axis=1)     # design matrix: [x_i, 1] per row
w = np.linalg.solve(A.T @ A, A.T @ y)      # the normal-equations closed-form "training step" (solve, not inv -- same reasoning as above)

print(f"True line:      y = {y_true_slope}x + {y_true_intercept}")
print(f"Recovered line: y = {w[0]:.3f}x + {w[1]:.3f}")

plt.figure(figsize=(6, 4))
plt.scatter(x, y, s=15, color="#4C72B0", label="noisy data")
plt.plot(x, A @ w, color="#DD8452", label="fit via normal equations")
plt.legend(); plt.title("Linear regression = solving an overdetermined system")
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Solve $\begin{cases}x+y=5\\2x-y=1\end{cases}$ by hand with elimination, then verify with
   `gaussian_elimination_solve`. As a sanity check, confirm the residual $A\mathbf{x}-\mathbf{b}$
   is exactly the zero vector — the defining property of an exact solution, not an
   approximation.

<details>
<summary>💡 Solution</summary>

Adding the two equations after doubling the first ($2x+2y=10$) and subtracting the second
($2x-y=1$) gives $3y=9\Rightarrow y=3$, then $x=5-y=2$. So $\mathbf{x}=(2,3)$, matching
`gaussian_elimination_solve` exactly. Check: $A\mathbf{x}-\mathbf{b} =
(1\cdot2+1\cdot3-5,\;2\cdot2-1\cdot3-1) = (0,0)$ — the residual is exactly zero, as it must be
for any exact solution of a consistent square system.

</details>

### 💭 UNDERSTAND
2. Two different overdetermined 3-equation, 2-unknown systems share the same first two
   equations, $x+y=3$ and $x-y=1$ (unique pairwise solution $x=2,y=1$). System A adds a third
   equation that is a linear combination of the first two ($2x=4$); System B instead adds
   $x+y=10$, which contradicts the first equation outright. Without solving anything yet,
   predict which system is consistent (has an exact solution satisfying all three equations) and
   which is not. Verify using $\text{rank}(A)$ vs. $\text{rank}([A\mid\mathbf{b}])$, then check
   each system's least-squares residual $\|A\mathbf{x}^\star-\mathbf{b}\|$ to confirm.

<details>
<summary>💡 Solution</summary>

System A is consistent, System B is not. For System A, $\text{rank}(A)=\text{rank}([A\mid
\mathbf{b}])=2$ (the third row adds no new information), and `lstsq` finds $\mathbf{x}^\star=
(2,1)$ with residual $\approx0$ (order $10^{-15}$, floating-point noise) — an exact solution
exists. For System B, $\text{rank}(A)=2$ but $\text{rank}([A\mid\mathbf{b}])=3$ — the augmented
column is *not* in the column space of $A$, so no exact solution exists; `lstsq` returns the
closest approximation, $\mathbf{x}^\star=(3.75,2.75)$, with a strictly positive residual
($\approx4.95$). The general rule (consistency, from the rank theorem): a system
$A\mathbf{x}=\mathbf{b}$ has an exact solution if and only if appending $\mathbf{b}$ as a column
doesn't increase $A$'s rank — extra equations are harmless if they're implied by the others, but
fatal to exactness if they add a genuinely new constraint that conflicts with the rest.

</details>

### ✏️ DERIVE
3. Derive the normal equations $A^\top A\,\mathbf{x}^\star = A^\top\mathbf{b}$ from the
   least-squares objective $f(\mathbf{x})=\|A\mathbf{x}-\mathbf{b}\|_2^2$ by expanding $f$ and
   setting its gradient to zero.

<details>
<summary>💡 Solution outline</summary>

Expand the squared norm as an inner product and multiply out:
$$f(\mathbf{x}) = (A\mathbf{x}-\mathbf{b})^\top(A\mathbf{x}-\mathbf{b}) =
\mathbf{x}^\top A^\top A\mathbf{x} - 2\mathbf{b}^\top A\mathbf{x} + \mathbf{b}^\top\mathbf{b}.$$
Using the standard gradient identities $\nabla_\mathbf{x}(\mathbf{x}^\top M\mathbf{x}) =
2M\mathbf{x}$ for symmetric $M$ (here $M=A^\top A$, which is always symmetric since
$(A^\top A)^\top=A^\top A$) and $\nabla_\mathbf{x}(\mathbf{c}^\top\mathbf{x})=\mathbf{c}$:
$$\nabla_\mathbf{x} f = 2A^\top A\,\mathbf{x} - 2A^\top\mathbf{b}.$$
Setting this to $\mathbf{0}$ and dividing by 2 gives exactly $A^\top A\,\mathbf{x}^\star =
A^\top\mathbf{b}$. This is a genuine minimum (not a saddle or maximum) because $f$'s Hessian is
$2A^\top A$, which is positive semi-definite for *any* $A$ (its eigenvalues, verified
numerically for the notebook's own $X$, are $\approx22.88$ and $\approx34.13$ — both
non-negative, as $A^\top A$'s eigenvalues always are). Numeric check: substituting the
notebook's own `w_normal_eq` back into $2X^\top Xw - 2X^\top y$ gives a vector of magnitude
$\approx4\times10^{-15}$ — zero up to floating-point rounding, confirming the closed form
actually zeroes the gradient it was derived from.

</details>

### 🐍 IMPLEMENT
4. Factor a random $4\times4$ matrix with `scipy.linalg.lu` (which returns $P,L,U$ such that
   $A=PLU$), and use the resulting factors to solve $A\mathbf{x}=\mathbf{b}$ for three different
   right-hand sides via two triangular solves
   (`scipy.linalg.solve_triangular`) instead of one general solve — first solving
   $L\mathbf{z}=P^\top\mathbf{b}$, then $U\mathbf{x}=\mathbf{z}$.

<details>
<summary>✅ How to know you're right</summary>

Your $\mathbf{x}$ must match `np.linalg.solve(A, b)` to floating-point precision for all three
right-hand sides. Two structural checks confirm you built the factorization correctly before
even solving anything: `P @ P.T` must equal the identity (a permutation matrix is always
orthogonal), and `P @ L @ U` must reconstruct the original $A$. If your triangular-solve results
disagree with `np.linalg.solve`, check that you used $P^\top\mathbf{b}$ (not $P\mathbf{b}$) as
the right-hand side for the first triangular solve — $P$ permutes rows of $A$, so undoing that
permutation on $\mathbf{b}$ requires $P^{-1}=P^\top$.

</details>

### 🤖 APPLY
5. A regression problem has more features than data points ($n_{\text{samples}}=5$,
   $n_{\text{features}}=8$), so $X^\top X$ is exactly singular (rank $\le5<8$). Solve for
   weights two ways: the plain normal equations $X^\top X\,\mathbf{w}=X^\top\mathbf{y}$, and
   ridge-regularized $(X^\top X+\lambda I)\mathbf{w}=X^\top\mathbf{y}$ for a small $\lambda$
   (e.g. $0.1$). Then perturb $\mathbf{y}$ by a *tiny* amount (noise of scale $10^{-8}$, far
   smaller than $\mathbf{y}$'s own scale) and re-solve both ways.

<details>
<summary>✅ What you should observe</summary>

`np.linalg.solve` doesn't raise an error on the exactly-singular $X^\top X$ (its condition
number is astronomically large, $\sim6\times10^{16}$, but not flagged as an outright failure)
— yet the *plain* solution is wildly sensitive: a relative perturbation to $\mathbf{y}$ of only
$\sim3\times10^{-9}$ produces a relative change in $\mathbf{w}$ of about $0.19$ (19%) — an
amplification of roughly $6\times10^7\times$. The ridge-regularized solution, by contrast,
barely moves under the same perturbation (relative change $\sim3\times10^{-9}$, i.e. an
amplification factor of about $1\times$ — essentially none). This is because adding $\lambda I$
raises every one of $X^\top X$'s eigenvalues by exactly $\lambda$, pulling the near-zero
eigenvalues (here, three of them are on the order of $10^{-15}$ or smaller — genuinely singular
directions) away from zero and collapsing the condition number from $\sim6\times10^{16}$ to
about $138$. The practical lesson mirrors why real training pipelines essentially never solve
the pure, unregularized normal equations when there are more parameters than data points: the
"exact" solution is numerically meaningless noise amplification, and a small amount of
regularization trades a little bit of fit (the ridge solution's own training residual is
$\approx0.10$, versus the plain solution's $\approx10^{-15}$) for a dramatically more stable
answer.

</details>

### 🚀 CHALLENGE
6. Build an underdetermined system (a $2\times4$ matrix $A$ and vector $\mathbf{b}$) and find
   the minimum-norm solution via `np.linalg.pinv(A) @ b`. Verify it satisfies
   $A\mathbf{x}=\mathbf{b}$, then generate other valid solutions by adding vectors from $A$'s
   null space (via `scipy.linalg.null_space`) and confirm they all have strictly larger norm
   than the pseudoinverse solution. Investigate *why*: what geometric relationship between the
   pseudoinverse solution and the null space guarantees this holds for literally any vector you
   add from the null space, not just the ones you happened to try?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms the pattern numerically first — for a sample $2\times4$ system, the
pseudoinverse solution has norm $\approx0.243$, while three other solutions built by adding
random null-space vectors have norms $\approx1.054$, $\approx0.892$, and $\approx0.525$, all
strictly larger — then explains *why* this is guaranteed, not coincidental: `pinv`'s solution
$\mathbf{x}_{\text{pinv}}$ is always orthogonal to $A$'s null space (every other valid solution
is $\mathbf{x}_{\text{pinv}}+\mathbf{v}$ for some null-space vector $\mathbf{v}$, since both
solve $A\mathbf{x}=\mathbf{b}$ and their difference must satisfy $A(\Delta\mathbf{x})=\mathbf{0}$
by linearity). Because $\mathbf{x}_{\text{pinv}}\perp\mathbf{v}$, the Pythagorean theorem applies
directly: $\|\mathbf{x}_{\text{pinv}}+\mathbf{v}\|^2 = \|\mathbf{x}_{\text{pinv}}\|^2 +
\|\mathbf{v}\|^2 > \|\mathbf{x}_{\text{pinv}}\|^2$ for *any* nonzero $\mathbf{v}$ in the null
space — verified numerically (both sides equal $\approx0.725$ for a sample perturbation) — which
is a strictly larger norm unless $\mathbf{v}=\mathbf{0}$. A strong answer connects this to why
minimum-norm solutions matter practically: among all weight vectors that fit training data
exactly (common in overparameterized models with more parameters than constraints), the
minimum-norm one is a specific, principled tie-breaker — the same bias toward "smallest"
solutions that motivates weight decay and ridge regularization in settings where an exact fit
isn't even required.

</details>

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 2 & Ch. 4 (Orthogonality and least squares)
- [scipy.linalg.lu documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.lu.html)

---

*Next notebook: [Determinants and Inverses](05_determinants_and_inverses.ipynb)*